In [1]:
import os
import sys

if "google.colab" in sys.modules:
    # Running in Colab

    !git clone https://github.com/pthengtr/kcw-analytics.git
    !cd /content/kcw-analytics && git pull origin main

    from google.colab import drive
    drive.mount("/content/drive")

    BASE_FOLDER = "/content/drive/Shareddrives"
    BASE_FOLDER_GIT = "/content"
else:
    # Running in local Jupyter
    BASE_FOLDER = r"G:\Shared drives"
    BASE_FOLDER_GIT = r"C:\Users\Windows 11\Notebook"

print("Using folder:", BASE_FOLDER)

Cloning into 'kcw-analytics'...
remote: Enumerating objects: 741, done.
remote: Counting objects: 100% (100/100), done.
remote: Compressing objects: 100% (77/77), done.
remote: Total 741 (delta 65), reused 39 (delta 23), pack-reused 641 (from 2)
Receiving objects: 100% (741/741), 625.45 KiB | 17.87 MiB/s, done.
Resolving deltas: 100% (498/498), done.
From https://github.com/pthengtr/kcw-analytics
 * branch            main       -> FETCH_HEAD
Already up to date.
Mounted at /content/drive
Using folder: /content/drive/Shareddrives


In [2]:
folder = f"{BASE_FOLDER}/KCW-Data/kcw_analytics/01_raw"

In [3]:
import os
import pandas as pd

data = {}

for file in os.listdir(folder):
    if file.endswith(".csv"):
        path = os.path.join(folder, file)
        data[file] = pd.read_csv(
            path,
            dtype={
              "BCODE": "string",
              "ITEMNO": "string",
              "BILLNO": "string",
            },
            encoding="utf-8-sig",
            low_memory=False   # stops chunk guessing
        )
        print(f"Loaded: {file} -> {data[file].shape}")



Loaded: raw_inventory_hq_2024.csv -> (4983, 8)
Loaded: raw_syp_pimas_purchase_bills.csv -> (3012, 49)
Loaded: raw_syp_simas_sales_bills.csv -> (13202, 49)
Loaded: raw_syp_pidet_purchase_lines.csv -> (28103, 41)
Loaded: raw_syp_sidet_sales_lines.csv -> (38554, 38)
Loaded: raw_hq_icmas_products.csv -> (114984, 94)
Loaded: raw_hq_pidet_purchase_lines.csv -> (154142, 41)
Loaded: raw_hq_pimas_purchase_bills.csv -> (50262, 49)
Loaded: raw_hq_sidet_sales_lines.csv -> (733804, 38)
Loaded: raw_hq_apmas_payable.csv -> (979, 20)
Loaded: raw_hq_armas_receivable.csv -> (2233, 20)
Loaded: raw_hq_simas_sales_bills.csv -> (276222, 49)
Loaded: raw_hq_pvmas_notes_vouchers.csv -> (13893, 32)
Loaded: raw_hq_rvmas_notes_vouchers.csv -> (13893, 32)


In [4]:


import sys
import importlib

# ensure repo is on path
repo_path = f"{BASE_FOLDER_GIT}/kcw-analytics"
if repo_path not in sys.path:
    sys.path.append(repo_path)

# import the module (NOT individual functions)
import src.kcw.utils as utils

# reload to pick up latest .py changes
importlib.reload(utils)

get_nonvat_sales_lines_last_purchase_vat = utils.get_nonvat_sales_lines_last_purchase_vat
audit_bcode_nonvat_sales_last_purchase_vat = utils.audit_bcode_nonvat_sales_last_purchase_vat

In [5]:
def filter_year_month(df, year, month, date_col="BILLDATE"):
    return df[pd.to_datetime(df[date_col]).dt.to_period("M") == f"{year}-{month:02d}"]

In [6]:
import pandas as pd

def filter_last_year(
    df: pd.DataFrame,
    date_col: str = "BILLDATE",
    *,
    years: int = 1,
    copy: bool = True,
    verbose: bool = True,
) -> pd.DataFrame:
    """
    Filter dataframe to keep rows within N years back from latest date.

    Parameters
    ----------
    df : pd.DataFrame
    date_col : str
        Column name containing date (default BILLDATE)
    years : int
        How many years back from latest date
    copy : bool
        Return a copy (safe for pipelines)
    verbose : bool
        Print diagnostics

    Returns
    -------
    pd.DataFrame
    """

    if date_col not in df.columns:
        raise ValueError(f"{date_col} not found in dataframe")

    # ensure datetime (legacy POS safe)
    dates = pd.to_datetime(df[date_col], errors="coerce")

    latest_date = dates.max()
    if pd.isna(latest_date):
        raise ValueError("No valid dates found")

    cutoff_date = latest_date - pd.DateOffset(years=years)

    mask = dates >= cutoff_date
    result = df.loc[mask]

    if copy:
        result = result.copy()

    if verbose:
        print(
            f"[filter_last_year] latest={latest_date.date()} | "
            f"cutoff={cutoff_date.date()} | rows={len(result):,}/{len(df):,}"
        )

    return result

In [7]:
import pandas as pd

def get_last_two_years_nonvat_sales_lines_last_purchase_vat(
    data,
    *,
    source: str,
    date_col: str = "BILLDATE",
    copy: bool = True,
    verbose: bool = True,
) -> pd.DataFrame:
    """
    Run get_nonvat_sales_lines_last_purchase_vat() for the latest year found in `data`
    and the previous year, then concat results.

    Assumes `data` is a dict-like object of dataframes (your KCW pattern).
    """

    # --- find any dataframe in `data` that has date_col
    candidate_years = []
    for _, obj in data.items():
        if isinstance(obj, pd.DataFrame) and date_col in obj.columns:
            y = pd.to_datetime(obj[date_col], errors="coerce").dt.year.max()
            if pd.notna(y):
                candidate_years.append(int(y))

    if not candidate_years:
        raise ValueError(f"Could not find any DataFrame in `data` with a valid {date_col}")

    latest_year = max(candidate_years)
    years = [latest_year - 1, latest_year]

    # --- loop and concat
    outs = []
    for y in years:
        out = get_nonvat_sales_lines_last_purchase_vat(data, year=y, source=source)
        out = out.copy() if copy else out
        out["YEAR"] = y  # optional but very useful
        outs.append(out)

    result = pd.concat(outs, ignore_index=True)

    if verbose:
        print(f"[last_two_years_nonvat] source={source} years={years} rows={len(result):,}")

    return result

In [8]:
nonvat_sales_lines_last_purchase_vat_hq = get_last_two_years_nonvat_sales_lines_last_purchase_vat(
    data, source="hq"
)

nonvat_sales_lines_last_purchase_vat_syp = get_last_two_years_nonvat_sales_lines_last_purchase_vat(
    data, source="syp"
)

[last_two_years_nonvat] source=hq years=[2025, 2026] rows=75,947
[last_two_years_nonvat] source=syp years=[2025, 2026] rows=16,393


In [9]:
nonvat_sales_lines_last_purchase_vat_hq = filter_last_year(nonvat_sales_lines_last_purchase_vat_hq, "BILLDATE")
nonvat_sales_lines_last_purchase_vat_syp = filter_last_year(nonvat_sales_lines_last_purchase_vat_syp, "BILLDATE")

[filter_last_year] latest=2026-03-13 | cutoff=2025-03-13 | rows=68,440/75,947
[filter_last_year] latest=2026-03-12 | cutoff=2025-03-12 | rows=16,393/16,393


In [10]:
mask = nonvat_sales_lines_last_purchase_vat_hq["BILLNO"].astype("string").str.contains("TF", na=False)

removed_tf = nonvat_sales_lines_last_purchase_vat_hq.loc[mask].copy()
nonvat_sales_lines_last_purchase_vat_hq = nonvat_sales_lines_last_purchase_vat_hq.loc[~mask].copy()

print(f"Removed TF/TFV lines: {len(removed_tf)}")

Removed TF/TFV lines: 12314


In [11]:
mask = nonvat_sales_lines_last_purchase_vat_hq["BCODE"].astype("string").str.startswith(("70", "91"), na=False)

removed_70 = nonvat_sales_lines_last_purchase_vat_hq.loc[mask].copy()
nonvat_sales_lines_last_purchase_vat_hq = nonvat_sales_lines_last_purchase_vat_hq.loc[~mask].copy()

print(f"Removed BCODE starting with 70 or 91: {len(removed_70)}")

Removed BCODE starting with 70 or 91: 338


In [12]:
mask = nonvat_sales_lines_last_purchase_vat_syp["BILLNO"].astype("string").str.contains("TF", na=False)

removed_tf = nonvat_sales_lines_last_purchase_vat_syp.loc[mask].copy()
nonvat_sales_lines_last_purchase_vat_syp = nonvat_sales_lines_last_purchase_vat_syp.loc[~mask].copy()

print(f"Removed TF/TFV lines: {len(removed_tf)}")

Removed TF/TFV lines: 0


In [13]:
mask = nonvat_sales_lines_last_purchase_vat_syp["BCODE"].astype("string").str.startswith(("70", "91"), na=False)

removed_70 = nonvat_sales_lines_last_purchase_vat_syp.loc[mask].copy()
nonvat_sales_lines_last_purchase_vat_syp = nonvat_sales_lines_last_purchase_vat_syp.loc[~mask].copy()

print(f"Removed BCODE starting with 70 or 91: {len(removed_70)}")

Removed BCODE starting with 70 or 91: 6


In [14]:
out_syp = nonvat_sales_lines_last_purchase_vat_syp
out_hq = nonvat_sales_lines_last_purchase_vat_hq

In [15]:
out_hq = filter_last_year(out_hq, "BILLDATE")
out_syp = filter_last_year(out_syp, "BILLDATE")

[filter_last_year] latest=2026-03-12 | cutoff=2025-03-12 | rows=55,788/55,788
[filter_last_year] latest=2026-03-12 | cutoff=2025-03-12 | rows=16,387/16,387


In [16]:
import pandas as pd

def split_negative_amount(
    df: pd.DataFrame,
    amount_col: str = "AMOUNT",
    *,
    copy: bool = True,
    verbose: bool = True,
):
    """
    Split dataframe into negative AMOUNT rows and non-negative rows.

    Returns
    -------
    df_negative, df_positive
    """

    if amount_col not in df.columns:
        raise ValueError(f"{amount_col} not found in dataframe")

    # ensure numeric (legacy POS safe)
    amount = pd.to_numeric(df[amount_col], errors="coerce")

    mask_neg = amount < 0

    df_negative = df.loc[mask_neg]
    df_positive = df.loc[~mask_neg]

    if copy:
        df_negative = df_negative.copy()
        df_positive = df_positive.copy()

    if verbose:
        print(
            f"[split_negative_amount] negative={len(df_negative):,} | "
            f"non_negative={len(df_positive):,} | total={len(df):,}"
        )

    return df_negative, df_positive


def join_po_from_simas(
    df_target: pd.DataFrame,
    df_simas: pd.DataFrame,
    *,
    key: str = "BILLNO",
    po_col: str = "PO",
    copy: bool = True,
    verbose: bool = True,
) -> pd.DataFrame:

    if key not in df_target.columns:
        raise ValueError(f"{key} not in df_target")
    if key not in df_simas.columns:
        raise ValueError(f"{key} not in df_simas")
    if po_col not in df_simas.columns:
        raise ValueError(f"{po_col} not in df_simas")

    # --- normalize join keys (non-destructive)
    tgt = df_target.copy()
    sim = df_simas.copy()

    tgt["_JOIN_KEY"] = tgt[key].astype("string").str.strip().str.upper()
    sim["_JOIN_KEY"] = sim[key].astype("string").str.strip().str.upper()

    # lookup table
    simas_lookup = (
        sim[["_JOIN_KEY", po_col]]
        .drop_duplicates(subset=["_JOIN_KEY"])
    )

    result = tgt.merge(
        simas_lookup,
        on="_JOIN_KEY",
        how="left"
    ).drop(columns=["_JOIN_KEY"])

    if copy:
        result = result.copy()

    if verbose:
        matched = result[po_col].notna().sum()
        print(f"[join_po_from_simas] matched PO rows: {matched:,}/{len(result):,}")

    return result


In [17]:
def process_branch_sales(out_df, data, branch):
    """
    branch: 'hq' or 'syp'
    """

    # 1) split negative lines
    out_neg, out_pos = split_negative_amount(out_df)

    # 3) get SIMAS reference file dynamically
    simas_key = f"raw_{branch}_simas_sales_bills.csv"
    df_simas = data[simas_key].copy()

    out_neg = join_po_from_simas(out_neg, df_simas)

    return out_pos, out_neg

In [18]:
out_hq_pos, out_hq_neg = process_branch_sales(out_hq, data, branch="hq")

out_syp_pos, out_syp_neg = process_branch_sales(out_syp, data, branch="syp")

[split_negative_amount] negative=1,140 | non_negative=54,648 | total=55,788
[join_po_from_simas] matched PO rows: 98/1,140
[split_negative_amount] negative=385 | non_negative=16,002 | total=16,387
[join_po_from_simas] matched PO rows: 21/385


In [23]:
out_hq_pos.columns

Index(['ID', 'JOURMODE', 'JOURTYPE', 'JOURDATE', 'BILLTYPE', 'BILLDATE',
       'BILLNO', 'LINE', 'ITEMNO', 'BCODE', 'PCODE', 'MCODE', 'DETAIL',
       'WHNUMBER', 'LOCATION1', 'STATUS', 'SERIAL', 'TAXIC', 'EXMPT', 'ISVAT',
       'QTY', 'UI', 'MTP', 'PRICE', 'XPRICE', 'DISCNT1', 'DISCNT2', 'DISCNT3',
       'DISCNT4', 'DED', 'VAT', 'AMOUNT', 'CHGAMT', 'ACCTNO', 'PAID',
       'ACCT_NO', 'DONE', 'CANCELED', 'LAST_PURCHASE_ISVAT', 'YEAR'],
      dtype='object')

In [24]:
pos_cols = ["BILLDATE","BILLNO", "BCODE", "QTY","MTP","UI","PRICE","AMOUNT"]

out_hq_pos_cleaned = out_hq_pos[pos_cols].copy()
out_syp_pos_cleaned = out_syp_pos[pos_cols].copy()

In [26]:
neg_cols = ["BILLDATE","BILLNO", "BCODE", "QTY","MTP","UI","PRICE","AMOUNT", "PO"]

out_hq_neg_cleaned = out_hq_neg[neg_cols].copy()
out_syp_neg_cleaned = out_syp_neg[neg_cols].copy()

In [30]:
def filter_year_fast(df, date_col, year):

    start = pd.Timestamp(year=year, month=1, day=1)
    end   = pd.Timestamp(year=year + 1, month=1, day=1)

    return df[
        (df[date_col] >= start) &
        (df[date_col] <  end)
    ].copy()

In [34]:
out_hq_pos_2026 = filter_year_fast(out_hq_pos_cleaned, "BILLDATE", 2026)
out_syp_pos_2026 = filter_year_fast(out_syp_pos_cleaned, "BILLDATE", 2026)
out_hq_neg_2026 = filter_year_fast(out_hq_neg_cleaned, "BILLDATE", 2026)
out_syp_neg_2026 = filter_year_fast(out_syp_neg_cleaned, "BILLDATE", 2026)

In [35]:
from datetime import datetime
import pytz

# Singapore timezone
tz = pytz.timezone("Asia/Bangkok")

today = datetime.now(tz)

YEAR = today.year
MONTH = today.month

print(YEAR, MONTH)

2026 3


In [36]:
import os

kcwdir = os.path.join(BASE_FOLDER, "KCW-Data")
print(kcwdir)

/content/drive/Shareddrives/KCW-Data


In [37]:
import os
from pathlib import Path

output_dir = Path(
    kcwdir,
    "kcw_analytics",
    "04_outputs",
    "TAR"
)
out_hq_pos_2026.to_csv(os.path.join(output_dir, "out_hq_pos.csv"), index=False, encoding="utf-8-sig")
out_hq_neg_2026.to_csv(os.path.join(output_dir, "out_hq_neg.csv"), index=False, encoding="utf-8-sig")

output_dir = Path(
    kcwdir,
    "kcw_analytics",
    "04_outputs",
    "3TAR"
)

out_syp_pos_2026.to_csv(os.path.join(output_dir, "out_syp_pos.csv"), index=False, encoding="utf-8-sig")
out_syp_neg_2026.to_csv(os.path.join(output_dir, "out_syp_neg.csv"), index=False, encoding="utf-8-sig")


In [38]:
out_hq_pos_2026.columns

Index(['BILLDATE', 'BILLNO', 'BCODE', 'QTY', 'MTP', 'UI', 'PRICE', 'AMOUNT'], dtype='object')